# Expanding $\varepsilon_{ijk}$ and contracting Kronecker deltas

This notebook demonstrates four derivation steps:

1. **`expand_eps`** — replace a rank-3 Levi-Civita symbol with its 6-term cofactor expansion in Kronecker deltas.
2. **`fold_sums`** — the inverse of `unroll_sums`: detect a concrete $N$-addend sum cycle and fold it back into an `ExplicitSum`.
3. **`contract_delta`** — simplify $\sum_m \delta^m{}_k\,\delta^m{}_l$ to $\delta_{kl}$.
4. **`contract_eps_pair`** — contract a pair of Levi-Civita symbols $\sum_i \varepsilon\,\varepsilon$ directly to the generalized Kronecker delta (see vibe 000035).


In [ ]:
import tender
import tender.derivation as td
from tender import Level, Realm
from IPython.display import Math, display

sp = tender.space_3d

## 1. `expand_eps`: the cofactor expansion

The Levi-Civita symbol in 3D can be written as a $3\times3$ determinant of Kronecker deltas:

$$
\varepsilon_{ijk}
= \det\!\begin{pmatrix}
    \delta^1{}_i & \delta^1{}_j & \delta^1{}_k \\
    \delta^2{}_i & \delta^2{}_j & \delta^2{}_k \\
    \delta^3{}_i & \delta^3{}_j & \delta^3{}_k
  \end{pmatrix}.
$$

The `expand_eps` step implements this as a 6-term (all $3!$ permutations) signed sum.

In [ ]:
ctx = tender.Context()
i, j, k = ctx.alloc_index(), ctx.alloc_index(), ctx.alloc_index()

eps = tender.levi_civita(
    Realm.Oblique, sp,
    [Level.Lower, Level.Lower, Level.Lower],
    [i, j, k], ctx=ctx)

display(Math(eps.latex()))

In [ ]:
expanded = td.expand_eps(eps)
display(Math(expanded.latex()))

## 2. Concrete evaluation: $\varepsilon_{123} = +1$

After `expand_eps`, applying `eval_delta_concrete` and `fold_arithmetic` reduces the expression to its scalar value.

In [ ]:
for indices, expected in [([1,2,3], 1), ([1,3,2], -1), ([1,1,2], 0)]:
    e = tender.levi_civita(Realm.Oblique, sp,
                           [Level.Lower]*3, indices)
    val = td.fold_arithmetic(td.eval_delta_concrete(td.expand_eps(e)))
    print(f"ε{indices} = {val.latex()}  (expected {expected})")

## 3. `fold_sums` + `contract_delta`

The completeness relation says $\sum_m \delta^m{}_k\,\delta^m{}_l = \delta_{kl}$.

Starting from the concrete sum $\delta^1{}_k\delta^1{}_l + \delta^2{}_k\delta^2{}_l + \delta^3{}_k\delta^3{}_l$, we:
1. Use **`fold_sums`** to recognise the 3-term cycle and introduce $\sum_m$.
2. Use **`contract_delta`** to contract the shared index $m$.

In [ ]:
ctx2 = tender.Context()
k2, l2 = ctx2.alloc_index(), ctx2.alloc_index()

def d(v, idx):
    return tender.delta(Realm.Oblique, sp, Level.Upper, Level.Lower, v, idx, ctx=ctx2)

concrete_sum = d(1,k2)*d(1,l2) + d(2,k2)*d(2,l2) + d(3,k2)*d(3,l2)
display(Math(concrete_sum.latex()))

In [ ]:
drv = td.Derivation(concrete_sum)
drv.step(td.fold_sums).step(td.contract_delta)

for step, expr in enumerate(drv.history):
    display(Math(rf"\text{{step {step}}}: {expr.latex()}"))

## 4. Full self-contraction of $\varepsilon_{ijk}$ at concrete indices

As a larger example, verify $\sum_{i,j,k} (\varepsilon_{ijk})^2 = 6$ using `expand_eps`
followed by the concrete evaluation pipeline.

In [ ]:
ctx3 = tender.Context()
ia, ja, ka = ctx3.alloc_index(), ctx3.alloc_index(), ctx3.alloc_index()

eps_sym = tender.levi_civita(
    Realm.Oblique, sp,
    [Level.Lower]*3, [ia, ja, ka], ctx=ctx3)

eps_sq = tender.explicit_sum(
    ia, tender.explicit_sum(
    ja, tender.explicit_sum(
    ka, eps_sym * eps_sym, ctx=ctx3), ctx=ctx3), ctx=ctx3)

drv2 = td.Derivation(eps_sq)
drv2.step(td.expand_eps)\
    .step(td.unroll_sums)\
    .step(td.eval_delta_concrete)\
    .step(td.fold_arithmetic)

print(f"Σ_ijk ε_ijk² = {drv2.current.latex()}")

## 5. `contract_eps_pair`: $\varepsilon\,\varepsilon$ as a generalized Kronecker delta

The product of two Levi-Civita symbols is a generalized Kronecker delta, so a
contraction over the shared indices closes in a **single symbolic step** — no
concrete-index unrolling, and no "creative" add/subtract step (see vibe 000035):

$$
\sum_i \varepsilon^{ijk}\varepsilon_{iml}
   = \delta^j_m\delta^k_l - \delta^j_l\delta^k_m,
\qquad
\sum_{ij} \varepsilon^{ijk}\varepsilon_{ijl} = 2\,\delta^k_l.
$$

The contracted indices are the nested `explicit_sum`s; `contract_eps_pair`
reads off the free slots and emits the Kronecker determinant directly.


**One shared index** — $\sum_i \varepsilon^{ijk}\varepsilon_{iml}$:

In [ ]:
ctx5 = tender.Context()
i5, j5, k5, m5, l5 = (ctx5.alloc_index() for _ in range(5))

imap5 = tender.IndexNameMap()
for idx, nm in [(i5, "i"), (j5, "j"), (k5, "k"), (m5, "m"), (l5, "l")]:
    imap5.assign(idx, nm)

ea = tender.levi_civita(Realm.Oblique, sp, [Level.Upper]*3, [i5, j5, k5], ctx=ctx5)
eb = tender.levi_civita(Realm.Oblique, sp, [Level.Lower]*3, [i5, m5, l5], ctx=ctx5)
contraction = tender.explicit_sum(i5, ea * eb, ctx=ctx5)

drv5 = td.Derivation(contraction, index_map=imap5)
drv5.step(td.contract_eps_pair)

for step in range(len(drv5.history)):
    display(Math(rf"\text{{step {step}}}:\quad {drv5.latex(step)}"))


**Two shared indices** — $\sum_{ij} \varepsilon^{ijk}\varepsilon_{ijl} = 2\,\delta^k_l$:

In [ ]:
ctx6 = tender.Context()
i6, j6, k6, l6 = (ctx6.alloc_index() for _ in range(4))

imap6 = tender.IndexNameMap()
for idx, nm in [(i6, "i"), (j6, "j"), (k6, "k"), (l6, "l")]:
    imap6.assign(idx, nm)

ea6 = tender.levi_civita(Realm.Oblique, sp, [Level.Upper]*3, [i6, j6, k6], ctx=ctx6)
eb6 = tender.levi_civita(Realm.Oblique, sp, [Level.Lower]*3, [i6, j6, l6], ctx=ctx6)
double = tender.explicit_sum(j6, tender.explicit_sum(i6, ea6 * eb6, ctx=ctx6), ctx=ctx6)

drv6 = td.Derivation(double, index_map=imap6)
drv6.step(td.contract_eps_pair)

display(Math(drv6.latex(0)  + r"\;=\;" + drv6.latex(-1)))
